# MGMT298D: Science and Strategy of AI
## Week 2 Assignment - Tree-Based Predictions
### Application: Vehicle Pricing

---

**Instructions:** Complete the exercises below by filling in the `???` placeholders and answering the questions in the designated cells. Run all code cells in order.

## Setup and Data Loading

We'll predict used Range Rover prices using three tree-based methods:
- Decision Trees
- Random Forests
- XGBoost (Gradient Boosting)

In [ ]:
import pandas as pd
import numpy as np
from sklearn import metrics
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv("https://raw.githubusercontent.com/ucla-anderson-SSAI/SSAI/refs/heads/main/range_rover.csv")
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)

print(f"Dataset: {len(df)} Range Rovers")
print(f"Training: {len(train_df)} | Test: {len(test_df)}")
df.head()

In [ ]:
# Quick data exploration
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(df['sellingprice'], bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Selling Price ($)')
axes[0].set_title(f'Price Distribution\nMean: ${df["sellingprice"].mean():,.0f}')

axes[1].scatter(df['year'], df['sellingprice'], alpha=0.5, s=20)
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Price ($)')
axes[1].set_title('Price vs Year')

axes[2].scatter(df['mileage'], df['sellingprice'], alpha=0.5, s=20, c='orange')
axes[2].set_xlabel('Mileage')
axes[2].set_ylabel('Price ($)')
axes[2].set_title('Price vs Mileage')

plt.tight_layout()
plt.show()

In [ ]:
# Data preparation: encode categorical variables
def prepare_data(df_train, df_test):
    train_enc = df_train.copy()
    test_enc = df_test.copy()
    encoders = {}
    
    for col in ['trim', 'state', 'color']:
        encoders[col] = LabelEncoder()
        combined = pd.concat([df_train[col], df_test[col]]).unique()
        encoders[col].fit(combined)
        train_enc[f'{col}_enc'] = encoders[col].transform(df_train[col])
        test_enc[f'{col}_enc'] = encoders[col].transform(df_test[col])
    
    feature_cols = ['year', 'mileage', 'trim_enc', 'state_enc', 'color_enc']
    return train_enc[feature_cols], test_enc[feature_cols], df_train['sellingprice'], df_test['sellingprice']

X_train, X_test, y_train, y_test = prepare_data(train_df, test_df)
print(f"Features: {list(X_train.columns)}")

---

## Model 1: Decision Tree

A decision tree makes predictions by learning simple if-then rules from the data.

In [ ]:
# ============================================================
# EXERCISE 1: Tune the decision tree depth
# ============================================================
# max_depth controls tree complexity. Try: 2, 5, 10, 20, None (unlimited)

MAX_DEPTH = ???  # <-- Fill in a value

dt_model = DecisionTreeRegressor(max_depth=MAX_DEPTH, min_samples_leaf=5, random_state=42)
dt_model.fit(X_train, y_train)

dt_train_pred = dt_model.predict(X_train)
dt_test_pred = dt_model.predict(X_test)

dt_train_mae = metrics.mean_absolute_error(y_train, dt_train_pred)
dt_test_mae = metrics.mean_absolute_error(y_test, dt_test_pred)

print(f"=== Decision Tree (max_depth={MAX_DEPTH}) ===")
print(f"Training MAE: ${dt_train_mae:,.0f}")
print(f"Test MAE:     ${dt_test_mae:,.0f}")
print(f"Overfitting Gap: ${dt_test_mae - dt_train_mae:,.0f}")

In [ ]:
# Visualize the decision tree (simplified)
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(dt_model, feature_names=list(X_train.columns), filled=True, 
          rounded=True, max_depth=3, fontsize=10, ax=ax)
plt.title(f'Decision Tree (showing first 3 levels)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Explore the effect of max_depth
depths = [2, 3, 5, 7, 10, 15, 20, None]
train_maes, test_maes = [], []

for d in depths:
    dt = DecisionTreeRegressor(max_depth=d, min_samples_leaf=5, random_state=42)
    dt.fit(X_train, y_train)
    train_maes.append(metrics.mean_absolute_error(y_train, dt.predict(X_train)))
    test_maes.append(metrics.mean_absolute_error(y_test, dt.predict(X_test)))

plt.figure(figsize=(10, 5))
x_labels = [str(d) if d else 'None' for d in depths]
plt.plot(x_labels, train_maes, 'b-o', label='Training MAE', linewidth=2)
plt.plot(x_labels, test_maes, 'r-o', label='Test MAE', linewidth=2)
plt.fill_between(x_labels, train_maes, test_maes, alpha=0.2, color='gray', label='Overfitting Gap')
plt.xlabel('Max Depth', fontsize=12)
plt.ylabel('MAE ($)', fontsize=12)
plt.title('Decision Tree: Overfitting as Depth Increases', fontsize=14)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Q1:** Looking at the plot above, what happens to the gap between training and test MAE as max_depth increases? What does this tell you about overfitting?

*Your answer:*


---

## Model 2: Random Forest

A Random Forest builds many trees on random subsets of data, then averages their predictions.

In [ ]:
# ============================================================
# EXERCISE 2: Tune the number of trees
# ============================================================
# n_estimators = number of trees. Try: 10, 50, 100, 200, 500

N_TREES = ???  # <-- Fill in a value

rf_model = RandomForestRegressor(n_estimators=N_TREES, max_depth=10, 
                                  min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_train_pred = rf_model.predict(X_train)
rf_test_pred = rf_model.predict(X_test)

rf_train_mae = metrics.mean_absolute_error(y_train, rf_train_pred)
rf_test_mae = metrics.mean_absolute_error(y_test, rf_test_pred)

print(f"=== Random Forest ({N_TREES} trees) ===")
print(f"Training MAE: ${rf_train_mae:,.0f}")
print(f"Test MAE:     ${rf_test_mae:,.0f}")
print(f"Overfitting Gap: ${rf_test_mae - rf_train_mae:,.0f}")

In [ ]:
# Explore effect of number of trees
tree_counts = [1, 5, 10, 25, 50, 100, 200]
rf_test_maes = []

for n in tree_counts:
    rf = RandomForestRegressor(n_estimators=n, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_test_maes.append(metrics.mean_absolute_error(y_test, rf.predict(X_test)))

plt.figure(figsize=(10, 5))
plt.plot(tree_counts, rf_test_maes, 'g-o', linewidth=2, markersize=8)
plt.xlabel('Number of Trees', fontsize=12)
plt.ylabel('Test MAE ($)', fontsize=12)
plt.title('Random Forest: More Trees Generally = Better (but diminishing returns)', fontsize=14)
plt.grid(alpha=0.3)
plt.show()

---

## Model 3: XGBoost (Gradient Boosting)

In [ ]:
# ============================================================
# EXERCISE 3: Tune the learning rate
# ============================================================
# learning_rate controls how fast the model learns. Try: 0.01, 0.05, 0.1, 0.3, 0.5

LEARNING_RATE = ???  # <-- Fill in a value

xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=LEARNING_RATE, 
                             max_depth=6, random_state=42)
xgb_model.fit(X_train, y_train)

xgb_train_pred = xgb_model.predict(X_train)
xgb_test_pred = xgb_model.predict(X_test)

xgb_train_mae = metrics.mean_absolute_error(y_train, xgb_train_pred)
xgb_test_mae = metrics.mean_absolute_error(y_test, xgb_test_pred)

print(f"=== XGBoost (learning_rate={LEARNING_RATE}) ===")
print(f"Training MAE: ${xgb_train_mae:,.0f}")
print(f"Test MAE:     ${xgb_test_mae:,.0f}")
print(f"Overfitting Gap: ${xgb_test_mae - xgb_train_mae:,.0f}")

In [ ]:
# Explore effect of learning rate
learning_rates = [0.01, 0.05, 0.1, 0.2, 0.3, 0.5]
xgb_train_maes, xgb_test_maes = [], []

for lr in learning_rates:
    xg = xgb.XGBRegressor(n_estimators=100, learning_rate=lr, max_depth=6, random_state=42)
    xg.fit(X_train, y_train)
    xgb_train_maes.append(metrics.mean_absolute_error(y_train, xg.predict(X_train)))
    xgb_test_maes.append(metrics.mean_absolute_error(y_test, xg.predict(X_test)))

plt.figure(figsize=(10, 5))
plt.plot(learning_rates, xgb_train_maes, 'b-o', label='Training MAE', linewidth=2)
plt.plot(learning_rates, xgb_test_maes, 'r-o', label='Test MAE', linewidth=2)
plt.xlabel('Learning Rate', fontsize=12)
plt.ylabel('MAE ($)', fontsize=12)
plt.title('XGBoost: Effect of Learning Rate', fontsize=14)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Q2:** What happens when the learning rate is too high (e.g., 0.5) vs. too low (e.g., 0.01)? What's the trade-off?

*Your answer:*


---

## Model Comparison

In [ ]:
# Final comparison with tuned models
results = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest', 'XGBoost'],
    'Train MAE': [dt_train_mae, rf_train_mae, xgb_train_mae],
    'Test MAE': [dt_test_mae, rf_test_mae, xgb_test_mae]
})
results['Overfitting Gap'] = results['Test MAE'] - results['Train MAE']

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MAE Comparison
x = np.arange(3)
width = 0.35
bars1 = axes[0].bar(x - width/2, results['Train MAE'], width, label='Training', color='steelblue')
bars2 = axes[0].bar(x + width/2, results['Test MAE'], width, label='Test', color='coral')
axes[0].set_xticks(x)
axes[0].set_xticklabels(results['Model'])
axes[0].set_ylabel('MAE ($)', fontsize=12)
axes[0].set_title('Model Comparison: Train vs Test MAE', fontsize=14)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Feature Importance (using Random Forest)
importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=True)

axes[1].barh(importance['Feature'], importance['Importance'], color='teal')
axes[1].set_xlabel('Importance', fontsize=12)
axes[1].set_title('Feature Importance (Random Forest)', fontsize=14)

plt.tight_layout()
plt.show()

print(results.round(0))

**Q3:** Based on the feature importance chart, which feature is most important for predicting Range Rover prices? Does this make sense from a business perspective?

*Your answer:*


---

## Question 4: Prediction Analysis

In [ ]:
# Analyze where the model does well vs poorly
test_results = test_df.copy()
test_results['predicted'] = rf_test_pred
test_results['error'] = test_results['sellingprice'] - test_results['predicted']
test_results['abs_error'] = np.abs(test_results['error'])

# Show best and worst predictions
print("=== Best Predictions (lowest error) ===")
print(test_results.nsmallest(5, 'abs_error')[['year', 'mileage', 'trim', 'sellingprice', 'predicted', 'error']])

print("\n=== Worst Predictions (highest error) ===")
print(test_results.nlargest(5, 'abs_error')[['year', 'mileage', 'trim', 'sellingprice', 'predicted', 'error']])

In [ ]:
# Error distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(test_results['error'], bins=30, edgecolor='black', alpha=0.7)
axes[0].axvline(x=0, color='red', linestyle='--')
axes[0].set_xlabel('Prediction Error ($)')
axes[0].set_title('Distribution of Errors\n(Positive = Underpredicted)')

axes[1].scatter(test_results['sellingprice'], test_results['predicted'], alpha=0.5)
max_val = max(test_results['sellingprice'].max(), test_results['predicted'].max())
axes[1].plot([0, max_val], [0, max_val], 'r--', label='Perfect Prediction')
axes[1].set_xlabel('Actual Price ($)')
axes[1].set_ylabel('Predicted Price ($)')
axes[1].set_title('Actual vs Predicted')
axes[1].legend()

plt.tight_layout()
plt.show()

**Q4:** Look at the worst predictions above. What patterns do you notice? Why might the model struggle with certain vehicles?

*Your answer:*


---

## Question 5: Model Selection for Business

**Q5a:** If you had to choose one model for a used car pricing tool, which would you pick and why? Consider both accuracy AND interpretability.

*Your answer:*


**Q5b:** A used car dealer asks: "How can I justify your price estimate to a skeptical customer?" Which model (Decision Tree, Random Forest, or XGBoost) would be easiest to explain to the customer? Why?

*Your answer:*


---

## Question 6: Real-World Considerations

**Q6:** The model was trained on Range Rover data only. A dealer asks if they can use it to price a BMW X5. What would you advise, and why? What would be needed to expand the model to other vehicles?

*Your answer:*
